# Lab: Mitigating harmful content in a recommender system

**Incident brief.** Your recommendation system is performing well on engagement and relevance metrics. However, internal monitoring has identified elevated **exposure to harmful content** in top-ranked recommendations. You have been asked to:

- diagnose the exposure problem,
- apply post-processing mitigation techniques **without retraining the model**, and
- justify relevance–safety trade-offs.

This lab simulates a trust & safety escalation on a live system. You will use the MovieLens 100k dataset with **synthetic** content risk labels — these are not moral judgements about the films, they represent a hypothetical platform policy.

**Time budget: ~45 minutes.** Work in pairs if you like, but each person submits their own reflections.

---

## Setup — given

Run the next two cells. These are not the interesting part of the lab.

In [2]:
from pathlib import Path
import urllib.request
import zipfile

url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = Path('ml-latest-small.zip')
dataset_dir = Path('ml-latest-small')

if not dataset_dir.exists():
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('.')

print(f'Dataset ready: {dataset_dir.resolve()}')

Dataset ready: /Users/Hari.Durai/Documents/AI_Apprenticeship/ResponsibleAI_Ethics/ml-latest-small


In [3]:
import pandas as pd
import numpy as np

ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies  = pd.read_csv('ml-latest-small/movies.csv')
print(ratings.shape, movies.shape)
ratings.head()

(100836, 4) (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Scenario setup — given

We simulate a deployed recommender by using **mean rating per movie** as the relevance score. We assign synthetic content risk labels based on genre.

Both of these are given — the interesting work starts after.

In [4]:
# Baseline relevance score per movie
movie_scores = ratings.groupby('movieId')['rating'].mean().reset_index()
movie_scores = movie_scores.rename(columns={'rating': 'base_score'})
data = movies.merge(movie_scores, on='movieId', how='left').dropna(subset=['base_score'])

# Synthetic content risk labels (hypothetical platform policy)
def label_content(genres):
    g = genres.split('|')
    if any(x in g for x in ['Horror', 'Thriller', 'Crime', 'War']):
        return 'harmful'
    if any(x in g for x in ['Action', 'Drama', 'Mystery']):
        return 'borderline'
    return 'safe'

data['content_label'] = data['genres'].apply(label_content)
data[['title', 'genres', 'base_score', 'content_label']].head()

,title,genres,base_score,content_label
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,safe
1,Jumanji (1995),Adventure|Children|Fantasy,3.431818,safe
2,Grumpier Old Men (1995),Comedy|Romance,3.259615,safe
3,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,borderline
4,Father of the Bride Part II (1995),Comedy,3.071429,safe


---

## Step 1 — Baseline exposure  (≈10 min)

**What you need to produce:**
1. A top-N ranking of movies by `base_score`.
2. The proportion of each `content_label` in that top-N.

**Decision points — write your answer in the markdown cell below the code:**
- **What value of N did you pick, and why?** (N = 10 is a default, but justify your choice — what does N represent in a real platform?)
- If harmful exposure in your top-N were presented on a dashboard, **what number would make you escalate?** Name it now, before you see the mitigation results.

**Hints:**
- `sort_values`, `head(N)`, `value_counts(normalize=True)` will get you there in 2–3 lines.

In [5]:
# Step 1: baseline top-N exposure
N = 20

baseline_topN = data.sort_values('base_score', ascending=False).head(N).copy()

baseline_exposure = (
    baseline_topN['content_label']
    .value_counts(normalize=True)
    .reindex(['harmful', 'borderline', 'safe'], fill_value=0)
 )

print(baseline_exposure)
baseline_topN[['title', 'base_score', 'content_label']]

content_label
harmful       0.25
borderline    0.30
safe          0.45
Name: proportion, dtype: float64


,title,base_score,content_label
7656,Paper Birds (Pájaros de papel) (2010),5.0,borderline
8107,"Act of Killing, The (2012)",5.0,safe
9083,Jump In! (2007),5.0,borderline
9094,Human (2015),5.0,safe
9096,L.A. Slasher (2015),5.0,harmful
4251,Lady Jane (1986),5.0,borderline
8154,Bill Hicks: Revelations (1993),5.0,safe
8148,Justice League: Doom (2012),5.0,borderline
4246,Open Hearts (Elsker dig for evigt) (2002),5.0,safe
9122,Formula of Love (1984),5.0,safe


**Your answers — Step 1:**

- N = 20 because it approximates the first page of recommendations across multiple rails (e.g., hero + carousels), so exposure is meaningful at product surface level.
- My pre-commitment escalation threshold for harmful exposure in top-N is 10% because anything above 1 in 10 prominent recommendations suggests a policy-level ranking issue, not just random noise.

---

## Step 2 — Mitigation 1: Post-processing filter  (≈5 min)

Remove all `harmful` items *before* ranking, then take the top-N.

**What you need to produce:**
1. The new top-N and its content_label breakdown.
2. The **relevance loss** vs baseline — e.g. how much does mean `base_score` in the top-N drop?

**Decision point:**
- This mitigation is binary — harmful is removed entirely. **What kinds of harm is this appropriate for, and what kinds is it not?**

In [6]:
# Step 2: hard filter harmful, then rank top-N
filter_topN = (
    data[data['content_label'] != 'harmful']
    .sort_values('base_score', ascending=False)
    .head(N)
    .copy()
)

filter_exposure = (
    filter_topN['content_label']
    .value_counts(normalize=True)
    .reindex(['harmful', 'borderline', 'safe'], fill_value=0)
)

# Relevance loss vs baseline top-N
baseline_mean_score = baseline_topN['base_score'].mean()
filter_mean_score = filter_topN['base_score'].mean()
relevance_loss = baseline_mean_score - filter_mean_score

print('Exposure:', filter_exposure.to_dict())
print(f'Mean score: baseline={baseline_mean_score:.3f}  filter={filter_mean_score:.3f}  loss={relevance_loss:.3f}')

Exposure: {'harmful': 0.0, 'borderline': 0.4, 'safe': 0.6}
Mean score: baseline=5.000  filter=5.000  loss=0.000


---

## Step 3 — Mitigation 2: Re-ranking by demotion  (≈5 min)

Instead of removing harmful content, **multiply** its score by a factor < 1 so it sinks in the ranking. Borderline content gets a milder demotion.

`new_score = base_score × demotion_factor(label)`

**Decision points — these matter more than the code:**
- **Pick your demotion factors.** What would you multiply `harmful` by? What about `borderline`? Note that 0 is equivalent to a filter and 1 is no change.
- **Justify your factors as a product decision, not a technical one.** A reviewer should be able to read your justification and understand the implied trade-off.

In [7]:
# Step 3: multiplicative demotion
def demotion_factor(label):
    if label == 'harmful':
        return 0.35
    if label == 'borderline':
        return 0.80
    return 1.0

data['rerank_score'] = data['base_score'] * data['content_label'].apply(demotion_factor)
rerank_topN = data.sort_values('rerank_score', ascending=False).head(N).copy()
rerank_exposure = (
    rerank_topN['content_label']
    .value_counts(normalize=True)
    .reindex(['harmful', 'borderline', 'safe'], fill_value=0)
)
rerank_mean_score = rerank_topN['base_score'].mean()

print('Exposure:', rerank_exposure.to_dict())
print(f'Mean score: {rerank_mean_score:.3f}')

Exposure: {'harmful': 0.0, 'borderline': 0.0, 'safe': 1.0}
Mean score: 5.000


**Your answers — Step 3:**

- Demotion factors: harmful = 0.35, borderline = 0.80
- Justification (one short paragraph): Harmful items receive a strong demotion to sharply reduce visibility while still allowing exceptional high-relevance edge cases to surface if they are clearly valuable. Borderline items receive a mild demotion to reduce cumulative risk without collapsing catalog diversity. This balances policy enforcement with user utility by avoiding an all-or-nothing suppression strategy.

---

## Step 4 — Mitigation 3: Penalty scoring  (≈5 min)

**Subtract** a fixed penalty from the score rather than multiplying it.

`new_score = base_score − penalty(label)`

**Decision points:**
- **Pick your penalties.** What do you subtract for `harmful`? For `borderline`?
- **How is this different in behaviour from Step 3's demotion?** When would penalty scoring do something meaningfully different from multiplication? (Hint: think about what happens to high-scoring vs low-scoring items.)

In [8]:
# Step 4: additive penalty scoring
PENALTY_HARMFUL = 1.10
PENALTY_BORDERLINE = 0.35

def penalty(label):
    return {'harmful': PENALTY_HARMFUL, 'borderline': PENALTY_BORDERLINE, 'safe': 0}[label]

data['penalty_score'] = data['base_score'] - data['content_label'].apply(penalty)
penalty_topN = data.sort_values('penalty_score', ascending=False).head(N).copy()
penalty_exposure = (
    penalty_topN['content_label']
    .value_counts(normalize=True)
    .reindex(['harmful', 'borderline', 'safe'], fill_value=0)
)
penalty_mean_score = penalty_topN['base_score'].mean()

print('Exposure:', penalty_exposure.to_dict())
print(f'Mean score: {penalty_mean_score:.3f}')

Exposure: {'harmful': 0.0, 'borderline': 0.0, 'safe': 1.0}
Mean score: 5.000


**Your answer — Step 4:**

- Penalties: harmful = 1.10, borderline = 0.35
- Difference in behaviour from multiplicative demotion: additive penalties hit all items in a label by the same absolute amount, so low-scoring harmful items drop very far while very high-scoring harmful items can still survive. Multiplicative demotion scales with original relevance, so it preserves relative gaps more strongly among high-score items.

---

## Step 5 — Compare  (≈10 min)

Build a comparison table: **Method | Harmful exposure | Borderline exposure | Mean score | Rank displacement**.

**Rank displacement** is a metric you have to define. One reasonable definition: *average change in rank position for the movies that were in the original top-N* — but that is not the only definition. You might care about how many **new** movies have been pulled into the top-N from outside it, or how far the displaced items fell.

**Decision points:**
- **Pick and define your rank displacement metric.** Write it down before you compute it.
- Which mitigation would you recommend in production, and why?
- Is there a case where you would recommend **combining** two of them rather than choosing?

In [9]:
# Step 5: comparison table with rank displacement
def build_rank_map(df, score_col):
    ranked = df.sort_values(score_col, ascending=False).reset_index(drop=True)
    ranked['rank'] = np.arange(1, len(ranked) + 1)
    return ranked.set_index('movieId')['rank']

base_rank_map = build_rank_map(data, 'base_score')

def rank_displacement(candidate_topN, candidate_score_col):
    cand_rank_map = build_rank_map(data, candidate_score_col)
    baseline_ids = baseline_topN['movieId'].tolist()
    disp = []
    for mid in baseline_ids:
        base_r = int(base_rank_map[mid])
        cand_r = int(cand_rank_map[mid])
        disp.append(abs(cand_r - base_r))
    return float(np.mean(disp))

comparison = pd.DataFrame({
    'method': ['baseline', 'filter', 'demotion', 'penalty'],
    'harmful_pct': [
        baseline_exposure['harmful'],
        filter_exposure['harmful'],
        rerank_exposure['harmful'],
        penalty_exposure['harmful'],
    ],
    'borderline_pct': [
        baseline_exposure['borderline'],
        filter_exposure['borderline'],
        rerank_exposure['borderline'],
        penalty_exposure['borderline'],
    ],
    'mean_score': [
        baseline_mean_score,
        filter_mean_score,
        rerank_mean_score,
        penalty_mean_score,
    ],
    'rank_displacement': [
        0.0,
        rank_displacement(filter_topN, 'base_score'),
        rank_displacement(rerank_topN, 'rerank_score'),
        rank_displacement(penalty_topN, 'penalty_score'),
    ],
})

comparison['harmful_pct'] = (comparison['harmful_pct'] * 100).round(1)
comparison['borderline_pct'] = (comparison['borderline_pct'] * 100).round(1)
comparison['mean_score'] = comparison['mean_score'].round(3)
comparison['rank_displacement'] = comparison['rank_displacement'].round(2)

comparison

,method,harmful_pct,borderline_pct,mean_score,rank_displacement
0,baseline,25.0,30.0,5.0,0.0
1,filter,0.0,40.0,5.0,0.0
2,demotion,0.0,0.0,5.0,1665.1
3,penalty,0.0,0.0,5.0,325.1


**Your answers — Step 5:**

- My definition of rank displacement: average absolute rank change (across the full catalog ranking) for movies that were in the baseline top-N.
- Recommended mitigation for production: additive penalty scoring.
- Reasoning (trade-off you are accepting): in this run, additive penalty achieved the same harmful-exposure reduction as demotion but with much lower rank displacement, so it better preserves ranking stability while still enforcing safety.
- Would you combine two? If yes, how and why? Yes. Use a hard filter for explicitly disallowed categories and additive penalties for borderline categories so severe violations are blocked while uncertain-risk content is de-prioritized rather than removed.

---

## Step 6 — Reflection  (≈10 min)

Write your answers in this cell. Bullet points are fine; short and specific beats long and generic.

1. **Which mitigation reduced harmful exposure most effectively?** By what margin?
2. **Which caused the largest relevance loss?** Does that matter?
3. **Who in your organisation should approve the thresholds you picked** (your N, your demotion factors, your penalties)? Why them and not someone else?
4. **When should a decision like this be escalated to human-in-the-loop oversight** rather than left to the automated mitigation? Name a concrete condition.
5. **What can this lab *not* tell you about the real system's safety?** (Think about what the synthetic labels are hiding, what top-N doesn't capture, what mean rating isn't measuring.)

---

*Your answers:*

1. All three mitigations reduced harmful exposure from 25% to 0% in top-N, a 25 percentage-point reduction; hard filtering is the strictest by design.
2. Relevance-loss by mean base score was negligible in this run (all methods stayed near 5.0), but rank displacement showed major quality/stability differences and therefore still matters.
3. Thresholds should be approved jointly by Trust & Safety, Product, and Legal/Policy leads, with Data Science advising on metric impact.
4. Escalate to human oversight when harmful exposure exceeds the pre-committed threshold for two consecutive monitoring windows, or when novel high-severity content types appear.
5. This lab uses synthetic labels and a simplified relevance proxy, so it cannot capture nuanced contextual harm, personalization effects, long-tail exposure, or downstream user outcomes.